# Item-Based Content Autoencoder Recommendation System

This notebook trains a deep Autoencoder using JAX and Flax on purely **Content Features** (Semantic Synopsis Embeddings + Categorical Tags). The model learns to compress these features into a dense, intelligent bottleneck embedding representation. We then use this learned embedding to compute similarities and recommend anime.

In [ ]:
try:
    import sentence_transformers
except ImportError:
    !pip install -q sentence-transformers

import os
import ast
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

import jax
import jax.numpy as jnp
from jax import random
import flax.linen as nn
from flax.training import train_state
import optax
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings('ignore')

### 1. Data Loading & Feature Extraction

In [ ]:
DATA_PATH = "/kaggle/input/datasets/neelagiriaditya/anime-dataset-jan-1917-to-oct-2025/datasets/details.csv"
df = pd.read_csv(DATA_PATH)

def parse_list(x):
    if pd.isna(x): return []
    if isinstance(x, str):
        x = x.strip()
        if x.startswith('[') and x.endswith(']'):
            try: return ast.literal_eval(x)
            except: pass
        return [i.strip() for i in x.split(',') if i.strip()]
    return []

df['genres'] = df['genres'].apply(parse_list)
df['themes'] = df['themes'].apply(parse_list)
df['demographics'] = df['demographics'].apply(parse_list)
df['synopsis'] = df['synopsis'].fillna("")
df['members'] = pd.to_numeric(df['members'], errors='coerce').fillna(0).astype(int)
df['mal_id'] = pd.to_numeric(df['mal_id'], errors='coerce').fillna(0).astype(int)

# Categorical Features
def combine_tags(row):
    tags = set()
    tags.update(row['genres'])
    tags.update(row['themes'])
    tags.update(row['demographics'])
    return list(tags)

df['meta_tags'] = df.apply(combine_tags, axis=1)
mlb = MultiLabelBinarizer()
categorical_matrix = mlb.fit_transform(df['meta_tags']).astype(np.float32)
print(f"Categorical feature shape: {categorical_matrix.shape}")

# Semantic Features
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    try:
        torch.ones(1, device="cuda")
    except Exception as e:
        print(f"CUDA is available but incompatible ({e}). Falling back to CPU.")
        device = "cpu"

print(f"Loading Sentence Transformer on {device.upper()}...")
model_st = SentenceTransformer('all-MiniLM-L6-v2', device=device)
semantic_matrix = model_st.encode(df['synopsis'].tolist(), show_progress_bar=True, batch_size=128)
print(f"Semantic feature shape: {semantic_matrix.shape}")

# Normalize semantic to same scale as binary categorical (approx)
semantic_matrix = (semantic_matrix - semantic_matrix.mean(axis=0)) / (semantic_matrix.std(axis=0) + 1e-6)

item_features = np.concatenate([categorical_matrix, semantic_matrix], axis=1)
print(f"Combined input shape: {item_features.shape}")

### 2. Autoencoder Architecture (JAX / Flax)

In [ ]:
class ContentAutoencoder(nn.Module):
    out_dim: int
    hidden_dim1: int = 1024
    hidden_dim2: int = 512
    bottleneck_dim: int = 256
    
    @nn.compact
    def __call__(self, x, training: bool = False):
        # Encoder
        h = nn.Dense(self.hidden_dim1)(x)
        h = nn.swish(h)
        if training:
            h = nn.Dropout(rate=0.2, deterministic=not training)(h)
            
        h = nn.Dense(self.hidden_dim2)(h)
        h = nn.swish(h)
        if training:
            h = nn.Dropout(rate=0.1, deterministic=not training)(h)
            
        bottleneck = nn.Dense(self.bottleneck_dim, name="bottleneck")(h)
        
        # Decoder
        h_dec = nn.Dense(self.hidden_dim2)(bottleneck)
        h_dec = nn.swish(h_dec)
        if training:
            h_dec = nn.Dropout(rate=0.1, deterministic=not training)(h_dec)
            
        h_dec = nn.Dense(self.hidden_dim1)(h_dec)
        h_dec = nn.swish(h_dec)
        if training:
            h_dec = nn.Dropout(rate=0.2, deterministic=not training)(h_dec)
            
        reconstruction = nn.Dense(self.out_dim)(h_dec)
        return reconstruction, bottleneck

class TrainState(train_state.TrainState):
    key: jax.Array

def create_train_state(rng, learning_rate, out_dim):
    model = ContentAutoencoder(out_dim=out_dim)
    dummy_input = jnp.ones((1, out_dim))
    params = model.init({"params": rng}, dummy_input)["params"]
    tx = optax.adam(learning_rate)
    return TrainState.create(apply_fn=model.apply, params=params, tx=tx, key=rng)

@jax.jit
def train_step(state, batch):
    dropout_rng, new_key = random.split(state.key)
    
    # Denoising: mask some inputs (30% noise for stronger correlation learning)
    mask = random.bernoulli(dropout_rng, p=0.7, shape=batch.shape)
    corrupted_batch = batch * mask
    
    def loss_fn(params):
        recon, _ = state.apply_fn({"params": params}, corrupted_batch, training=True, rngs={"dropout": dropout_rng})
        loss = jnp.mean(optax.l2_loss(recon, batch))
        return loss
        
    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    state = state.apply_gradients(grads=grads)
    state = state.replace(key=new_key)
    return state, loss

### 3. Training Loop

In [ ]:
BATCH_SIZE = 256
EPOCHS = 80
LEARNING_RATE = 1e-3
OUT_DIM = item_features.shape[1]

rng = random.PRNGKey(42)
num_samples = item_features.shape[0]
num_batches = int(np.ceil(num_samples / BATCH_SIZE))

# Cosine learning rate decay schedule
lr_schedule = optax.cosine_decay_schedule(
    init_value=LEARNING_RATE,
    decay_steps=EPOCHS * num_batches
)

state = create_train_state(rng, lr_schedule, OUT_DIM)

print("Starting Autoencoder Training...")
pbar = tqdm(range(EPOCHS), desc="Training Autoencoder")
for epoch in pbar:
    indices = np.random.permutation(num_samples)
    epoch_losses = []
    
    for i in range(0, num_samples, BATCH_SIZE):
        batch_idx = indices[i:i+BATCH_SIZE]
        batch_x = jnp.array(item_features[batch_idx])
        state, loss = train_step(state, batch_x)
        epoch_losses.append(loss)
        
    mean_loss = np.mean(epoch_losses)
    pbar.set_postfix(loss=f"{mean_loss:.5f}")
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {mean_loss:.5f}")

# Save model weights for backend use
from flax import serialization
out_dir = "/kaggle/working" if os.path.exists("/kaggle") else "."
os.makedirs(out_dir, exist_ok=True)
weights_path = os.path.join(out_dir, "content_ae_weights.msgpack")

with open(weights_path, "wb") as f:
    f.write(serialization.to_bytes(state.params))
print(f"Saved model weights to {weights_path}")

### 4. Extracting Learned Embeddings & Recommendations

In [ ]:
print("Extracting bottleneck embeddings...")
_, bottlenecks = state.apply_fn({"params": state.params}, jnp.array(item_features), training=False)
learned_embeddings = np.array(bottlenecks)

print("Computing learned cosine similarity matrix...")
learned_sim = cosine_similarity(learned_embeddings)
np.fill_diagonal(learned_sim, 0.0)

def get_recommendations(user_anime_list, top_k=20, popularity_weight=0.15):
    valid_indices = df[df['mal_id'].isin(user_anime_list)].index.tolist()
    if not valid_indices:
        print("No valid anime found from the user's list.")
        return [], []
        
    user_profile_sim = learned_sim[valid_indices].mean(axis=0)
    
    log_members = np.log1p(df['members'].values)
    max_log_members = log_members.max() if log_members.max() > 0 else 1.0
    popularity_scores = log_members / max_log_members
    
    combined_scores = (1.0 - popularity_weight) * user_profile_sim + popularity_weight * popularity_scores
    
    masked_scores = combined_scores.copy()
    masked_scores[valid_indices] = -np.inf
    
    top_indices = np.argsort(masked_scores)[-top_k:][::-1]
    top_scores = masked_scores[top_indices]
    anime_ids = df.iloc[top_indices]['mal_id'].values
    return anime_ids, top_scores

sample_user_list = [20, 269]
print("Sample Recommendations for User Profile:")
recs, scores = get_recommendations(sample_user_list)
for rank, (aid, score) in enumerate(zip(recs, scores), 1):
    print(f"{rank}. Anime ID: {aid} (Score: {score:.3f})")